In [69]:
import pandas as pd
import unicodedata
from rapidfuzz import process, fuzz

In [70]:
supabase = pd.read_csv('supabase_ds.csv')
transfermarket = pd.read_csv('transfermarket_ds.csv')

In [71]:
# --- Text Normalization ---
def normalizza_testo(testo):
    if pd.isna(testo): return ""
    testo = str(testo).lower().strip()
    return "".join(c for c in unicodedata.normalize('NFD', testo) if unicodedata.category(c) != 'Mn')

# Prepariamo le colonne pulite per il match
supabase['nome_clean'] = supabase['nome'].apply(normalizza_testo)
supabase['club_clean'] = supabase['club'].apply(normalizza_testo)

transfermarket['lastname_clean'] = transfermarket['name'].apply(normalizza_testo)
transfermarket['club_clean'] = transfermarket['current_club_name'].apply(normalizza_testo)


# ==========================================
# STEP 1: BRIDGE KEY PER LE SQUADRE
# ==========================================
print("Esecuzione Step 1: Matching Squadre...")

squadre_supa = supabase['club_clean'].unique()
squadre_tm = transfermarket['club_clean'].unique()

lista_bridge_club = []
bridge_club_id = 1

for club_s in squadre_supa:
    # Cerchiamo il club corrispondente su TM
    match = process.extractOne(club_s, squadre_tm, scorer=fuzz.partial_ratio)
    
    # Creiamo il legame se troviamo un match decente. 
    # N.B. Squadre come "Svincolato" o "Diablitos FC" potrebbero non trovare match su TM.
    tm_matched_club = match[0] if (match and match[1] > 60) else None
    
    lista_bridge_club.append({
        'supa_club_clean': club_s,
        'tm_club_clean': tm_matched_club,
        'bridge_club_key': bridge_club_id
    })
    bridge_club_id += 1

df_bridge_club = pd.DataFrame(lista_bridge_club)

# Iniettiamo la bridge_club_key in Supabase
supabase = supabase.merge(
    df_bridge_club[['supa_club_clean', 'bridge_club_key']], 
    left_on='club_clean', right_on='supa_club_clean', how='left'
).drop(columns=['supa_club_clean'])

# Iniettiamo la bridge_club_key in Transfermarkt
# Usiamo drop_duplicates per evitare di moltiplicare le righe se più squadre Supa matchano la stessa squadra TM per errore
df_bridge_club_tm = df_bridge_club.dropna(subset=['tm_club_clean']).drop_duplicates(subset=['tm_club_clean'])
transfermarket = transfermarket.merge(
    df_bridge_club_tm[['tm_club_clean', 'bridge_club_key']], 
    left_on='club_clean', right_on='tm_club_clean', how='left'
).drop(columns=['tm_club_clean'])


# ==========================================
# STEP 2: BRIDGE KEY PER I GIOCATORI (REVISIONATO)
# ==========================================
print("Esecuzione Step 2: Matching Giocatori con Soglia di Sicurezza...")

lista_bridge_player = []
bridge_player_id = 1

# Impostiamo una soglia minima di sicurezza
# 80 è un ottimo compromesso di partenza per il token_set_ratio
SOGLIA_MATCH = 65

for index, row in supabase.iterrows():
    supa_id = row['id']
    b_club_key = row['bridge_club_key']
    nome_giocatore = row['nome_clean']
    
    # 1. Filtriamo TM per la stessa chiave squadra
    tm_candidates = transfermarket[transfermarket['bridge_club_key'] == b_club_key]
    
    tm_code_selezionato = None
    
    # 2. Cerchiamo il match all'interno della squadra
    if not tm_candidates.empty:
        scelte = tm_candidates['lastname_clean'].tolist()
        
        # CAMBIO CRITICO: Usiamo token_set_ratio invece di ratio
        miglior_match = process.extractOne(nome_giocatore, scelte, scorer=fuzz.token_set_ratio)
        
        # CAMBIO CRITICO: Accettiamo il match SOLO se supera la soglia
        if miglior_match and miglior_match[1] >= SOGLIA_MATCH:
            riga_tm = tm_candidates[tm_candidates['lastname_clean'] == miglior_match[0]].iloc[0]
            tm_code_selezionato = riga_tm['player_code']
    
    # Creiamo il record ponte. 
    # Se il punteggio era < 80, tm_code_selezionato resterà None.
    # L'ID Supabase viene comunque registrato per garantire la cardinalità 1,1 lato tuo.
    lista_bridge_player.append({
        'supabase_id': supa_id,
        'tm_player_code': tm_code_selezionato,
        'bridge_player_key': bridge_player_id
    })
    bridge_player_id += 1

df_bridge_player = pd.DataFrame(lista_bridge_player)

# Iniettiamo la bridge_player_key in Supabase (Garantita 1,1)
supabase = supabase.merge(
    df_bridge_player[['supabase_id', 'bridge_player_key']], 
    left_on='id', right_on='supabase_id', how='left'
).drop(columns=['supabase_id'])

# Iniettiamo la bridge_player_key in Transfermarkt (Sarà 0,1: i non matchati avranno NaN)
# Rimuoviamo i NaN da tm_player_code per non inquinare i dati di TM
df_bridge_player_tm = df_bridge_player.dropna(subset=['tm_player_code'])
transfermarket = transfermarket.merge(
    df_bridge_player_tm[['tm_player_code', 'bridge_player_key']], 
    left_on='player_code', right_on='tm_player_code', how='left'
).drop(columns=['tm_player_code'])

# Pulizia colonne temporanee
supabase.drop(columns=['nome_clean', 'club_clean'], inplace=True, errors='ignore')
transfermarket.drop(columns=['lastname_clean', 'club_clean'], inplace=True, errors='ignore')

print("Matching completato con successo!")
print(supabase[['id', 'nome', 'club', 'bridge_club_key', 'bridge_player_key']].head(10))

Esecuzione Step 1: Matching Squadre...
Esecuzione Step 2: Matching Giocatori con Soglia di Sicurezza...
Matching completato con successo!
    id      nome        club  bridge_club_key  bridge_player_key
0  907   Mosconi       Inter                1                  1
1  902  Vermesan      Verona                2                  2
2   44  Fullkrug       Milan                3                  3
3   50  Borrelli    Cagliari                4                  4
4   34  Kilicsoy    Cagliari                4                  5
5  896     Lahdo        Como                5                  6
6  897    Bakola    Sassuolo                6                  7
7  870  Bozhinov        Pisa                7                  8
8  305      Faye   Cremonese                8                  9
9  904     Balbo  Fiorentina                9                 10


In [72]:
# Uniamo i dati originali di Supabase con i dati utili di Transfermarkt
# usiamo la 'bridge_player_key' come cerniera.
df_check = supabase.merge(
    transfermarket[['bridge_player_key', 'name', 'current_club_name', 'url']],
    on='bridge_player_key',
    how='left' # Mantiene tutti i giocatori Supabase, anche quelli senza match
)

# Rinominiamo le colonne per affiancarle e renderle leggibili
df_check = df_check.rename(columns={
    'nome': 'Giocatore_Supa',
    'club': 'Club_Supa',
    'name': 'Giocatore_TM',
    'current_club_name': 'Club_TM'
})

# Selezioniamo solo le colonne che ci interessano per il check visivo
colonne_vista = ['id', 'Giocatore_Supa', 'Giocatore_TM', 'Club_Supa', 'Club_TM']
df_vista = df_check[colonne_vista]

# 1. VISUALIZZA I MATCH AVVENUTI CON SUCCESSO
print("--- MATCH TROVATI ---")
match_successo = df_vista[df_vista['Giocatore_TM'].notna()]
print(match_successo.head(20))
print(f"\nTotale match trovati: {len(match_successo)}")

# 2. VISUALIZZA I MATCH FALLITI (Utile per ricalibrare le soglie o trovare anomalie)
print("\n--- MATCH FALLITI (Nessun riscontro su Transfermarkt) ---")
match_falliti = df_vista[df_vista['Giocatore_TM'].isna()]
print(match_falliti[['id', 'Giocatore_Supa', 'Club_Supa']].head(20))
print(f"\nTotale match falliti: {len(match_falliti)}")

--- MATCH TROVATI ---
     id Giocatore_Supa             Giocatore_TM   Club_Supa  \
2    44       Fullkrug          Niclas Füllkrug       Milan   
3    50       Borrelli         Gennaro Borrelli    Cagliari   
5   896          Lahdo             Adrian Lahdo        Como   
6   897         Bakola            Darryl Bakola    Sassuolo   
7   870       Bozhinov           Rosen Bozhinov        Pisa   
8   305           Faye             Mikayil Faye   Cremonese   
9   904          Balbo               Luis Balbo  Fiorentina   
10  874         Lirola               Pol Lirola      Verona   
11  335    Lykogiannis  Charalampos Lykogiannis     Bologna   
12  879    Zatterstrom         Nils Zätterström       Genoa   
13  304   Coulibaly W.           Woyo Coulibaly    Sassuolo   
14  337          Vitik             Martin Vitík     Bologna   
15  423      Kossounou         Odilon Kossounou    Atalanta   
16  398           Hien                Isak Hien    Atalanta   
17  417        Circati       Ales

In [73]:
match_falliti.to_csv('match_falliti.csv', index=False)
match_successo.to_csv('match_successo.csv', index=False)

In [76]:
transfermarket.columns

Index(['first_name', 'last_name', 'name', 'last_season', 'player_code',
       'country_of_birth', 'date_of_birth', 'sub_position', 'position', 'foot',
       'height_in_cm', 'contract_expiration_date', 'image_url',
       'international_caps', 'international_goals', 'url', 'current_club_name',
       'market_value_in_eur', 'highest_market_value_in_eur', 'bridge_club_key',
       'bridge_player_key'],
      dtype='object')